# 按照蛋白链对齐，计算蛋白和多肽分别的rmsd，合并预测数据到csv文件中

In [1]:
import os
import json
import numpy as np
import pandas as pd

OUTPUT_ROOT = "./af3_test"
REF_PDB_DIR = "./PepSet_dimer"
BACKBONE_SET = {"N", "CA", "C", "O"}
ATOM_ORDER = {"N": 0, "CA": 1, "C": 2, "O": 3}


def parse_pdb_backbone(pdb_path):
    chains = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            atom_name = line[12:16].strip()
            if atom_name not in BACKBONE_SET:
                continue
            chain_id = line[21:22]
            try:
                res_seq = int(line[22:26].strip())
            except ValueError:
                continue
            x = float(line[30:38].strip())
            y = float(line[38:46].strip())
            z = float(line[46:54].strip())
            chains.setdefault(chain_id, []).append((res_seq, atom_name, x, y, z))

    result = {}
    for cid, atoms in chains.items():
        atoms.sort(key=lambda a: (a[0], ATOM_ORDER[a[1]]))
        result[cid] = np.array([[x, y, z] for _, _, x, y, z in atoms], dtype=float)
    return result


def parse_cif_backbone(cif_path):
    col_map = {}
    chains = {}
    with open(cif_path) as f:
        lines = f.readlines()

    atom_start = -1
    for i, line in enumerate(lines):
        if line.strip() == "loop_" and i + 1 < len(lines) and lines[i + 1].strip().startswith("_atom_site."):
            atom_start = i
            break

    if atom_start == -1:
        return chains

    j = atom_start + 1
    while j < len(lines) and lines[j].strip().startswith("_atom_site."):
        col_name = lines[j].strip().split()[0]
        col_map[col_name] = len(col_map)
        j += 1

    # 兼容 label_XXX (AF3) 和 auth_XXX (ESMFold) 两种列名
    idx_chain = col_map.get("_atom_site.auth_asym_id") or col_map.get("_atom_site.label_asym_id", 6)
    idx_atom = col_map.get("_atom_site.label_atom_id") or col_map.get("_atom_site.auth_atom_id", 12)
    idx_seq = col_map.get("_atom_site.auth_seq_id") or col_map.get("_atom_site.label_seq_id", 9)
    idx_x = col_map.get("_atom_site.Cartn_x", 14)
    idx_y = col_map.get("_atom_site.Cartn_y", 15)
    idx_z = col_map.get("_atom_site.Cartn_z", 16)
    idx_model = col_map.get("_atom_site.pdbx_PDB_model_num", 17)

    for line in lines[j:]:
        parts = line.strip().split()
        if not parts or parts[0] != "ATOM":
            continue
        if idx_model < len(parts) and parts[idx_model] != "1":
            continue
        atom_name = parts[idx_atom]
        if atom_name not in BACKBONE_SET:
            continue
        chain_id = parts[idx_chain]
        try:
            res_seq = int(parts[idx_seq])
        except (ValueError, IndexError):
            continue
        x = float(parts[idx_x])
        y = float(parts[idx_y])
        z = float(parts[idx_z])
        chains.setdefault(chain_id, []).append((res_seq, atom_name, x, y, z))

    result = {}
    for cid, atoms in chains.items():
        atoms.sort(key=lambda a: (a[0], ATOM_ORDER[a[1]]))
        result[cid] = np.array([[x, y, z] for _, _, x, y, z in atoms], dtype=float)
    return result


def kabsch_align(P, Q):
    """
    Kabsch: 求 R 和 t 使 P @ R.T + t ≈ Q.
    P, Q: (N, 3). 返回 (R, t, rmsd).
    """
    p_cent = P.mean(axis=0)
    q_cent = Q.mean(axis=0)
    Pc = P - p_cent
    Qc = Q - q_cent

    H = Pc.T @ Qc
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        R = Vt.T @ U.T

    t = q_cent - p_cent @ R.T
    P_rot = Pc @ R.T
    rmsd = float(np.sqrt(np.mean(np.sum((P_rot - Qc) ** 2, axis=1))))
    return R, t, rmsd


complexes = sorted([
    d for d in os.listdir(OUTPUT_ROOT)
    if os.path.isdir(os.path.join(OUTPUT_ROOT, d))
])

for pdb in complexes:
    ref_pdb_path = os.path.join(REF_PDB_DIR, f"{pdb}.pdb")
    complex_output = os.path.join(OUTPUT_ROOT, pdb)

    if not os.path.exists(ref_pdb_path):
        print(f"[SKIP] {pdb}: reference PDB not found")
        continue

    ref_chains = parse_pdb_backbone(ref_pdb_path)
    if "L" not in ref_chains:
        print(f"[SKIP] {pdb}: no L chain in reference")
        continue
    ref_pep = ref_chains["L"]
    ref_prot = None
    for cid, coords in ref_chains.items():
        if cid != "L":
            ref_prot = coords
            break
    if ref_prot is None:
        print(f"[SKIP] {pdb}: no non-L chain in reference")
        continue

    rows = []
    sample_dirs = sorted([
        d for d in os.listdir(complex_output)
        if d.startswith("seed-") and os.path.isdir(os.path.join(complex_output, d))
    ])

    for sample_dir in sample_dirs:
        sample_path = os.path.join(complex_output, sample_dir)

        # 解析 seed-XX_sample-XX 格式 (AF3 下划线分割)
        dir_parts = sample_dir.split("_")
        try:
            seed = int(dir_parts[0].split("-")[1])
            sample_id = int(dir_parts[1].split("-")[1])
        except (IndexError, ValueError):
            continue

        # 读取 confidences.json
        conf_name = f"{pdb}_{sample_dir}_confidences.json"
        conf_path = os.path.join(sample_path, conf_name)
        if not os.path.exists(conf_path):
            continue
        try:
            with open(conf_path) as f:
                conf = json.load(f)
            atom_chain_ids = conf["atom_chain_ids"]
            atom_plddts = conf["atom_plddts"]
        except (KeyError, json.JSONDecodeError):
            continue

        # 按链聚合 pLDDT 均值（全原子平均）
        chain_plddt = {}
        for cid, plddt in zip(atom_chain_ids, atom_plddts):
            chain_plddt.setdefault(cid, []).append(plddt)
        if "A" not in chain_plddt or "B" not in chain_plddt:
            continue
        chain_plddt_A = round(float(np.mean(chain_plddt["A"])), 4)
        chain_plddt_B = round(float(np.mean(chain_plddt["B"])), 4)
        plddt_mean = round((chain_plddt_A + chain_plddt_B) / 2, 4)

        # 读取 summary_confidences.json
        json_name = f"{pdb}_{sample_dir}_summary_confidences.json"
        json_path = os.path.join(sample_path, json_name)
        if not os.path.exists(json_path):
            continue
        try:
            with open(json_path) as f:
                metrics = json.load(f)
            ptm = round(metrics["ptm"], 4)
            iptm = round(metrics["iptm"], 4)
            ranking_score = round(metrics["ranking_score"], 4)
        except (KeyError, json.JSONDecodeError):
            continue

        # 读取结构坐标算 RMSD
        cif_name = f"{pdb}_{sample_dir}_model.cif"
        cif_path = os.path.join(sample_path, cif_name)
        if not os.path.exists(cif_path):
            continue
        pred_chains = parse_cif_backbone(cif_path)
        if "A" not in pred_chains or "B" not in pred_chains:
            continue
        pred_prot = pred_chains["A"]
        pred_pep = pred_chains["B"]

        if len(ref_prot) != len(pred_prot):
            continue

        # 用蛋白链求 R 和 t
        R, t, protein_rmsd = kabsch_align(pred_prot, ref_prot)
        protein_rmsd = round(protein_rmsd, 4)

        # 多肽施加相同的 R 和 t
        if len(ref_pep) == 0 or len(pred_pep) == 0:
            peptide_rmsd = None
        elif len(ref_pep) != len(pred_pep):
            peptide_rmsd = None
        else:
            pred_pep_aligned = pred_pep @ R.T + t
            diffs = pred_pep_aligned - ref_pep
            peptide_rmsd = round(float(np.sqrt(np.mean(np.sum(diffs ** 2, axis=1)))), 4)

        rows.append({
            "complex": pdb,
            "seed": seed,
            "id": sample_id,
            "chain_plddt_mean_A": chain_plddt_A,
            "chain_plddt_mean_B": chain_plddt_B,
            "plddt_mean": plddt_mean,
            "ptm": ptm,
            "iptm": iptm,
            "ranking_score": ranking_score,
            "protein_rmsd": protein_rmsd,
            "peptide_rmsd": peptide_rmsd,
        })

    if rows:
        result = pd.DataFrame(rows)
        result.to_csv(os.path.join(complex_output, "metrics_summary.csv"), index=False, na_rep="NaN")
        print(f"[OK] {pdb}: {len(rows)} samples, saved to metrics_summary.csv")
    else:
        print(f"[EMPTY] {pdb}: no valid samples")

[OK] 1a0n: 15 samples, saved to metrics_summary.csv


In [5]:
# 合并所有的csv文件,存放在/home/junjiechen/1_work/250401-Dpepalign/Benchmark/esmfold2/PepSet_dimer_output_esmfold2下
import pandas as pd
import os
rows = []
for dir in sorted(os.listdir("PepSet_dimer_output_esmfold2_fast")):
    for file in sorted(os.listdir(os.path.join("PepSet_dimer_output_esmfold2_fast", dir))):
        if file.endswith("metrics_summary.csv"):
            df = pd.read_csv(os.path.join("PepSet_dimer_output_esmfold2_fast", dir, file))
            rows.append(df)
if rows:
    df_combined = pd.concat(rows, ignore_index=True)
else:
    raise ValueError("No metrics_summary.csv files found in the directory.")

df_combined.to_csv('./results/esmfold2_fast_metrics_summary_loop10_samples200.csv', index=False)



In [6]:
# 检查summary文件，计算每个样本的成功率。成功率的定义为每个complex存在一个多肽plddt>=70，iptm>=0.7，多肽rmsd<2.5的样本，则这个样本预测成功
import pandas as pd
df = pd.read_csv('results/esmfold2_fast_metrics_summary_loop10_samples200.csv')
success_count = 0

for complex_name, group in df.groupby('complex'):
    if any(
        (group['chain_plddt_mean_B'] >= 70) &
        (group['iptm'] >= 0.7) &
        (group['peptide_rmsd'] < 2.5)
    ):
        success_count += 1

total_count = df['complex'].nunique()
success_rate = success_count / total_count if total_count > 0 else 0

print(f"Oracle Success count: {success_count}")
print(f"Total count: {total_count}")
print(f"Oracle Success rate: {success_rate:.2%}")

Oracle Success count: 0
Total count: 168
Oracle Success rate: 0.00%


# DockQ评估蛋白多肽复合物结构预测